# PSO-parallel — Informe Final

**Entrega 2 · Concurrencia/Paralelización**

---

## Resumen ejecutivo

Se implementa Particle Swarm Optimization (PSO) canónico con cinco estrategias de evaluación del fitness:
secuencial (V0), hilos (V1), procesos (V2), asyncio (V3) y NumPy vectorizado (V4).
Los experimentos se ejecutan sobre cuatro funciones benchmark estándar
(Sphere, Rosenbrock, Rastrigin, Ackley) en dimensiones d=2, d=10 y d=30.

**Conclusión principal:** V4 (NumPy vectorizado) es la estrategia más rápida para funciones CPU-bound,
V3 (asyncio) es óptima para evaluaciones I/O-bound con latencia asimétrica,
y V1/V2 presentan overhead neto para funciones de evaluación barata
debido al GIL y al coste de serialización IPC respectivamente.

In [ ]:
import os, sys, json, csv
from collections import defaultdict
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

RESULTS_DIR = "results"
LABELS = {
    "sequential": "V0 Sequential",
    "threading": "V1 Threading",
    "multiprocessing": "V2 Multiprocessing",
    "asyncio": "V3 Asyncio",
    "numpy": "V4 NumPy",
}

def load_records(d):
    recs = []
    if not os.path.isdir(d):
        return recs
    for e in sorted(os.listdir(d)):
        p = os.path.join(d, e, "summary.json")
        if os.path.isfile(p):
            with open(p) as f:
                data = json.load(f)
            recs.append({**data["config"], **data["result"]})
    return recs

def load_histories(d):
    groups = defaultdict(list)
    if not os.path.isdir(d):
        return dict(groups)
    for e in sorted(os.listdir(d)):
        sp = os.path.join(d, e, "summary.json")
        hp = os.path.join(d, e, "history.csv")
        if not (os.path.isfile(sp) and os.path.isfile(hp)):
            continue
        with open(sp) as f:
            cfg = json.load(f)["config"]
        h = [float(r["best_fitness"]) for r in csv.DictReader(open(hp))]
        groups[(cfg["objective"], cfg["dim"], cfg["evaluator"])].append(h)
    return dict(groups)

records = load_records(RESULTS_DIR)
histories = load_histories(RESULTS_DIR)
evaluators = sorted({r["evaluator"] for r in records})
objectives = sorted({r["objective"] for r in records})
dims = sorted({r["dim"] for r in records})
print(f"{len(records)} experimentos | evaluadores: {evaluators}")
print(f"objetivos: {objectives} | dimensiones: {dims}")

---
## 1. Metodología experimental

### 1.1 Algoritmo PSO

Se implementa PSO canónico con topología **global-best** y criterio de parada doble:
límite de iteraciones (`max_iters=500`) y estancamiento (`patience=50` iteraciones sin mejora > `tol=1e-8`).
La estrategia de límites es **ClampBounds**: la posición se recorta a `[lo, hi]` y
la velocidad se pone a cero en las dimensiones que colisionan con la pared.
Los parámetros por defecto son `w=0.7`, `c1=1.5`, `c2=1.5`, `n_particles=30`.

### 1.2 Funciones benchmark

| Función | Bounds | Óptimo | Dificultad |
|---|---|---|---|
| Sphere | [-5.12, 5.12] | 0 | Unimodal, convexa |
| Rosenbrock | [-2.048, 2.048] | 0 | Valle estrecho |
| Rastrigin | [-5.12, 5.12] | 0 | Multimodal (muchos mínimos locales) |
| Ackley | [-32.768, 32.768] | 0 | Multimodal con mínimo global casi plano |

Dimensiones evaluadas: **d=2, d=10, d=30**. Seeds: 42, 43, 44 (3 repeticiones por configuración).

### 1.3 Estrategias de evaluación

| Versión | Estrategia | Caso de uso óptimo |
|---|---|---|
| V0 | Secuencial | Referencia baseline |
| V1 | ThreadPoolExecutor | Funciones que liberan el GIL (NumPy pesado, I/O) |
| V2 | ProcessPoolExecutor + batching | Funciones CPU-bound muy costosas (>10 ms/partícula) |
| V3 | asyncio.gather + run_in_executor | Funciones I/O-bound con latencia asimétrica |
| V4 | NumPy batch (n×d → n) | Funciones vectorizables, enjambres grandes |

### 1.4 Protocolo experimental

- Medición de tiempos con `time.perf_counter`.
- Se instrumentan por separado: `time_eval` (evaluación del fitness) y `time_update` (actualización de partículas).
- Resultados guardados en `results/<objetivo>_d<dim>_s<seed>_<evaluador>/summary.json` + `history.csv`.
- Cada `summary.json` incluye git hash y datos de hardware para reproducibilidad.

---
## 2. Resultados

In [ ]:
# Tabla: tiempo medio (s) por evaluador, objetivo y dimensión
if not records:
    print("Sin resultados. Ejecuta run_benchmarks.py primero.")
else:
    header = f"{'objetivo':<14} {'d':>4}  " + "  ".join(f"{LABELS.get(e, e):>18}" for e in evaluators)
    print("=== Tiempo medio (s) ===")
    print(header)
    print("-" * len(header))
    for obj in objectives:
        for dim in dims:
            row = [f"{obj:<14} {dim:>4}"]
            for ev in evaluators:
                times = [r["time_total"] for r in records
                         if r["objective"] == obj and r["dim"] == dim and r["evaluator"] == ev]
                row.append(f"{np.mean(times):>8.3f} ±{np.std(times):>5.3f}" if times else f"{'—':>18}")
            print("  ".join(row))

In [ ]:
# Tabla: speedup vs V0 (sequential)
if records:
    header = f"{'objetivo':<14} {'d':>4}  " + "  ".join(f"{LABELS.get(e, e):>18}" for e in evaluators)
    print("=== Speedup vs V0 ===")
    print(header)
    print("-" * len(header))
    for obj in objectives:
        for dim in dims:
            base = [r["time_total"] for r in records
                    if r["objective"] == obj and r["dim"] == dim and r["evaluator"] == "sequential"]
            if not base:
                continue
            base_mean = np.mean(base)
            row = [f"{obj:<14} {dim:>4}"]
            for ev in evaluators:
                times = [r["time_total"] for r in records
                         if r["objective"] == obj and r["dim"] == dim and r["evaluator"] == ev]
                speedup = base_mean / np.mean(times) if times else None
                row.append(f"{speedup:>17.2f}x" if speedup else f"{'—':>18}")
            print("  ".join(row))

In [ ]:
# Curvas de convergencia promedio (media ± std sobre seeds)
if histories:
    obj_list = sorted({k[0] for k in histories})
    dim_list = sorted({k[1] for k in histories})
    ev_list = sorted({k[2] for k in histories})
    colors = plt.cm.tab10.colors

    for obj in obj_list:
        fig, axes = plt.subplots(1, len(dim_list), figsize=(5 * len(dim_list), 4), sharey=False)
        if len(dim_list) == 1:
            axes = [axes]
        for ax, dim in zip(axes, dim_list):
            for i, ev in enumerate(ev_list):
                hs = histories.get((obj, dim, ev), [])
                if not hs:
                    continue
                maxlen = max(len(h) for h in hs)
                arr = np.array([h + [h[-1]] * (maxlen - len(h)) for h in hs])
                mean, std = arr.mean(0), arr.std(0)
                iters = np.arange(maxlen)
                c = colors[i % len(colors)]
                ax.semilogy(iters, mean, label=LABELS.get(ev, ev), color=c)
                ax.fill_between(iters, np.clip(mean - std, 1e-15, None), mean + std, alpha=0.15, color=c)
            ax.set_title(f"d={dim}")
            ax.set_xlabel("Iteración")
            ax.set_ylabel("Best fitness")
            ax.grid(True, which="both", ls="--", alpha=0.4)
            ax.legend(fontsize=7)
        fig.suptitle(f"{obj} — curvas de convergencia (media ± std)")
        plt.tight_layout()
        plt.savefig(f"{RESULTS_DIR}/convergence_{obj}.png", dpi=100)
        plt.show()
        print(f"Guardado: {RESULTS_DIR}/convergence_{obj}.png")
else:
    print("Sin historiales. Ejecuta run_benchmarks.py primero.")

In [ ]:
# Boxplots: distribución del fitness final por evaluador
if records:
    fig, axes = plt.subplots(1, len(objectives), figsize=(4 * len(objectives), 5))
    if len(objectives) == 1:
        axes = [axes]
    for ax, obj in zip(axes, objectives):
        data, labels_bp = [], []
        for ev in evaluators:
            fits = [r["best_fitness"] for r in records if r["objective"] == obj and r["evaluator"] == ev]
            if fits:
                data.append(fits)
                labels_bp.append(LABELS.get(ev, ev))
        ax.boxplot(data, tick_labels=labels_bp, patch_artist=True)
        ax.set_yscale("log")
        ax.set_title(obj)
        ax.set_ylabel("Best fitness")
        ax.tick_params(axis="x", rotation=20)
        ax.grid(True, axis="y", ls="--", alpha=0.4)
    plt.suptitle("Distribución del fitness final por evaluador (todas las dimensiones y seeds)")
    plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/boxplot_fitness.png", dpi=100)
    plt.show()

---
## 3. Discusión crítica

### 3.1 V1 — Threading y el GIL

Python ejecuta un único hilo a la vez mediante el Global Interpreter Lock (GIL).
Para las funciones benchmark (Sphere, Rastrigin…), cuya evaluación es puramente Python/NumPy,
los hilos no se ejecutan en paralelo real: compiten por el GIL, añadiendo overhead de
sincronización sin beneficio. El resultado es que **V1 es sistemáticamente más lento que V0**
en estas funciones.

V1 sería beneficioso en dos escenarios:
- La función objetivo llama a código C que libera el GIL (NumPy con operaciones costosas).
- La evaluación involucra I/O de red o disco (el hilo cede el GIL durante la espera).

### 3.2 V2 — Multiprocessing e IPC

Cada worker de `ProcessPoolExecutor` corre en un proceso separado con su propio intérprete,
eliminando el GIL. Sin embargo, cada evaluación requiere:
1. **Serialización (pickle)** de la posición `NDArray` → bytes.
2. **Transferencia IPC** (pipe/socket) al worker.
3. Ejecución de la función.
4. Serialización del resultado float → bytes y transferencia de vuelta.

Para funciones baratas (Sphere en d=2: < 0.01 ms), el overhead de IPC (~0.1–1 ms por tarea)
domina y V2 es **5–20× más lento** que V0. El **batching** mitiga esto agrupando
partículas en lotes, reduciendo el número de round-trips, pero no elimina el overhead base.

V2 proporciona speedup real cuando el tiempo de evaluación por partícula supera ~10 ms.

### 3.3 V3 — Asyncio y concurrencia cooperativa

Asyncio no elimina el GIL ni crea procesos: es concurrencia cooperativa dentro de un único
hilo. Su ventaja aparece cuando la evaluación está bloqueada esperando I/O.
`asyncio.gather` lanza todas las corrutinas simultáneamente; mientras una espera,
el event loop ejecuta las otras.

La función `noisy_sphere` simula este escenario: cada evaluación duerme entre 5 y 50 ms
(latencia aleatoria de un servicio externo). Con evaluación secuencial:
tiempo ≈ Σ latencias ≈ 750 ms (30 partículas × 25 ms promedio).
Con asyncio: tiempo ≈ max(latencias) ≈ 50 ms. **Speedup teórico ~15×.**

Para funciones CPU-bound, V3 es equivalente a V1 (ambos usan hilos, ambos bloquean el GIL).

### 3.4 V4 — NumPy vectorizado (paralelismo implícito)

V4 apila las posiciones de todas las partículas en una matriz `(n, d)` y evalúa
en una sola llamada NumPy. Las operaciones matriciales se delegan a **BLAS/LAPACK**
(OpenBLAS, MKL según la instalación), que pueden usar SIMD y múltiples núcleos internamente.

Ventajas frente a V0:
- Elimina el bucle Python sobre partículas (interpretado, lento).
- Mejor aprovechamiento de caché CPU al procesar datos contiguos en memoria.
- Sin overhead de threading, IPC ni event loop.

**Limitación:** solo se vectoriza la evaluación del fitness. La actualización de velocidades
y posiciones sigue siendo un bucle Python (para mantener el núcleo PSO común e intercambiable).
Para dimensiones grandes (d=30, n=30), la fase de actualización representa un porcentaje
creciente del tiempo total.

### 3.5 Trade-offs por dimensión y coste de evaluación

| Escenario | Estrategia recomendada |
|---|---|
| Función barata, cualquier d | **V4 (NumPy)** |
| Función I/O-bound (latencia variable) | **V3 (asyncio)** |
| Función muy costosa CPU (>10 ms) | **V2 (multiprocessing)** |
| Función que libera el GIL | **V1 (threading)** |
| Prototipado / depuración | **V0 (sequential)** |

In [ ]:
# Grid search: mejores configuraciones por objetivo
import glob
grid_dir = os.path.join(RESULTS_DIR, "grid_search")
grid_files = glob.glob(os.path.join(grid_dir, "*.json"))

if not grid_files:
    print("Sin resultados de grid search. Ejecuta run_grid_search.py primero.")
else:
    for gf in sorted(grid_files):
        with open(gf) as f:
            gr = json.load(f)
        name = os.path.basename(gf).replace("_grid.json", "")
        print(f"\n=== {name} — top 3 configuraciones ===")
        print(f"{'w':>6} {'c1':>6} {'c2':>6} {'n':>5} {'fitness':>14} {'auc':>14} {'conv_iter':>10}")
        for r in gr[:3]:
            auc = r.get('mean_auc', float('nan'))
            ci = r.get('mean_conv_iter', float('nan'))
            print(f"{r['w']:>6.2f} {r['c1']:>6.2f} {r['c2']:>6.2f} {r['n_particles']:>5d}"
                  f" {r['mean_fitness']:>14.4e} {auc:>14.4e} {ci:>10.1f}")

---
## 4. Recomendaciones

1. **Usar V4 (NumPy) por defecto** para cualquier función benchmark estándar.
   Es la estrategia más rápida sin overhead adicional y no requiere configurar workers ni event loops.

2. **Usar V3 (asyncio) cuando la función objetivo consulta servicios externos**
   (REST, simuladores, bases de datos) con latencia variable.
   Configurar `max_concurrent` si el servicio tiene límite de conexiones simultáneas.

3. **Evitar V1 y V2 para funciones baratas** (< 1 ms por partícula).
   El overhead de threading/IPC supera siempre el tiempo de evaluación.
   V2 solo vale la pena si la evaluación individual cuesta > 10 ms.

4. **En dimensiones altas (d=30)**, el cuello de botella se desplaza
   hacia la actualización de velocidades (bucle Python).
   Una mejora futura sería vectorizar también la fase de actualización
   reformulando el enjambre como matrices `(n, d)`.

5. **Para el grid search**, la métrica más informativa es `mean_auc`
   (área bajo la curva de convergencia): penaliza tanto la lentitud de convergencia
   como la calidad del resultado final, siendo más robusta que el fitness final aislado.

---
## 5. Conclusión

No existe una estrategia universalmente mejor: la elección óptima depende del coste
de evaluación de la función objetivo y de si ese coste es CPU-bound o I/O-bound.
Para el caso más común en benchmarks de optimización (funciones analíticas baratas),
la vectorización NumPy (V4) supera a las estrategias basadas en paralelismo de tareas
(V1, V2, V3) porque elimina el overhead de coordinación y aprovecha las optimizaciones
de bajo nivel de BLAS.